In [1]:
import pandas as pd

In [2]:
# value = 6 - log2(num_teams_remaining)
STAGE_LOOKUP = {"group": 0.42, "r32": 1, "r16": 2, "quarter": 3, "semi": 4, "third": 4.42, "final": 5, "winner": 6}

In [3]:
raw_scraped = pd.read_csv("01_scraped.csv")

In [4]:
player_lookup = raw_scraped[["player_id", "name"]].drop_duplicates().sort_values(["name", "player_id"])
duplicate_names = player_lookup["name"].duplicated(keep=False)
player_lookup["safe_name"] = player_lookup["name"]
player_lookup.loc[duplicate_names, "safe_name"] = (
    player_lookup.loc[duplicate_names, "name"] + " (" + player_lookup.loc[duplicate_names, "player_id"].str[:3] + ")"
)
player_lookup.to_csv("02_player_lookup.csv", index=False)

In [5]:
groups_raw = raw_scraped[raw_scraped["stage"] == "group"]
groups_raw["tip_id"] = "GROUP-" + groups_raw["home"].str.replace(" ", "_").str.cat(
    groups_raw["away"].str.replace(" ", "_"), sep="-"
)

In [6]:
groups_raw["result"] = 0  # draws
groups_raw.loc[groups_raw["h_score"] > groups_raw["a_score"], "result"] = 1  # home wins
groups_raw.loc[groups_raw["h_score"] < groups_raw["a_score"], "result"] = -1  # away wins

In [7]:
group_tips = groups_raw.set_index(["player_id", "tip_id"])["result"].sort_index()

In [8]:
ko_raw = raw_scraped[raw_scraped["stage"] != "group"]

In [9]:
ko_raw["stage_val"] = ko_raw["stage"].map(STAGE_LOOKUP).astype(float)
ko_raw.sort_values("stage_val")
ko_raw["loser"] = ko_raw["home"]
ko_raw.loc[ko_raw["h_score"] > ko_raw["a_score"], "loser"] = ko_raw["away"]
ko_raw["loser"] = ko_raw["loser"].str.replace("&", "and").str.replace(" ", "_").str.replace(".", "")

In [10]:
elimination_raw = (
    ko_raw[ko_raw["stage"] != "third"].sort_values("stage_val").drop_duplicates(["player_id", "loser"], keep="last")
)
elimination_raw["tip_id"] = "BRACKET-" + elimination_raw["loser"]

elimination_tips = elimination_raw.set_index(["player_id", "tip_id"])["stage_val"]

In [11]:
winners_raw = ko_raw[(ko_raw["stage"] == "third") | (ko_raw["stage"] == "final")]

winners_raw["winner"] = winners_raw["home"]
winners_raw.loc[winners_raw["home"] == winners_raw["loser"], "winner"] = winners_raw["away"]

winners_raw.loc[winners_raw["stage"] == "final", "stage_val"] = STAGE_LOOKUP["winner"]

winners_raw["tip_id"] = "BRACKET-" + winners_raw["winner"]


winners_tips = winners_raw.set_index(["player_id", "tip_id"])["stage_val"]

In [12]:
bracket_tips_raw = pd.concat([elimination_tips, winners_tips])

bracket_tips = (
    bracket_tips_raw[~bracket_tips_raw.index.duplicated(keep="last")].unstack(fill_value=STAGE_LOOKUP["group"]).stack()
)

In [13]:
all_teams = set(groups_raw["home"]).union(groups_raw["away"])
bracket_teams = set(ko_raw["home"]).union(ko_raw["away"])
never_made_it_out_of_the_group = all_teams - bracket_teams
always_eliminated_tip_ids = sorted(
    "BRACKET-"
    + pd.Series(sorted(never_made_it_out_of_the_group))
    .str.replace("&", "and")
    .str.replace(" ", "_")
    .str.replace(".", "")
)

In [14]:
all_tips = pd.concat([group_tips, bracket_tips]).unstack()
all_tips.loc[:, always_eliminated_tip_ids] = 0.42
all_tips = all_tips.sort_index(axis="columns")

In [15]:
all_tips.to_csv("02_tips_matrix.csv")